In [1]:
import polars as pl

from datetime import date

from ohlc_dss_model.data import load_parquet, remove_incomplete_days, intraday_session_tagging, session_tagging, filter_valid_sessions

from ohlc_dss_model.features import aggregate_sessions, yang_zhang, HISTORICAL_SPEC, TODAY_SPEC

from ohlc_dss_model.utils import convert_to_timezone

from ohlc_dss_model import config

In [2]:
df = load_parquet()
df = convert_to_timezone(df)
df = session_tagging(df)
df = intraday_session_tagging(df)
df = remove_incomplete_days(df)

print(df.head(5))

shape: (5, 8)
┌──────────────────┬─────────┬─────────┬─────────┬─────────┬────────┬────────────┬─────────────────┐
│ DateTime         ┆ Open    ┆ High    ┆ Low     ┆ Close   ┆ Volume ┆ Session    ┆ Intraday_Sessio │
│ ---              ┆ ---     ┆ ---     ┆ ---     ┆ ---     ┆ ---    ┆ ---        ┆ n               │
│ datetime[ns, Ame ┆ f64     ┆ f64     ┆ f64     ┆ f64     ┆ u64    ┆ date       ┆ ---             │
│ rica/New_York]   ┆         ┆         ┆         ┆         ┆        ┆            ┆ str             │
╞══════════════════╪═════════╪═════════╪═════════╪═════════╪════════╪════════════╪═════════════════╡
│ 2016-01-03       ┆ 4592.5  ┆ 4606.75 ┆ 4592.5  ┆ 4603.25 ┆ 1134   ┆ 2016-01-04 ┆ Asia            │
│ 18:00:00 EST     ┆         ┆         ┆         ┆         ┆        ┆            ┆                 │
│ 2016-01-03       ┆ 4603.0  ┆ 4603.0  ┆ 4597.5  ┆ 4600.5  ┆ 425    ┆ 2016-01-04 ┆ Asia            │
│ 18:30:00 EST     ┆         ┆         ┆         ┆         ┆        ┆        

In [3]:
features = aggregate_sessions(df)
print(features.head(5))

shape: (5, 13)
┌────────────┬──────────┬────────────┬─────────┬───┬─────────┬──────────┬────────────┬─────────┐
│ Session    ┆ O_London ┆ O_New York ┆ O_Asia  ┆ … ┆ L_Asia  ┆ C_London ┆ C_New York ┆ C_Asia  │
│ ---        ┆ ---      ┆ ---        ┆ ---     ┆   ┆ ---     ┆ ---      ┆ ---        ┆ ---     │
│ date       ┆ f64      ┆ f64        ┆ f64     ┆   ┆ f64     ┆ f64      ┆ f64        ┆ f64     │
╞════════════╪══════════╪════════════╪═════════╪═══╪═════════╪══════════╪════════════╪═════════╡
│ 2016-01-04 ┆ 4529.75  ┆ 4498.75    ┆ 4592.5  ┆ … ┆ 4522.25 ┆ 4498.25  ┆ 4506.25    ┆ 4529.5  │
│ 2016-01-05 ┆ 4509.0   ┆ 4494.75    ┆ 4505.5  ┆ … ┆ 4498.25 ┆ 4494.75  ┆ 4481.5     ┆ 4509.25 │
│ 2016-01-06 ┆ 4445.0   ┆ 4396.25    ┆ 4482.0  ┆ … ┆ 4425.5  ┆ 4396.25  ┆ 4448.5     ┆ 4444.5  │
│ 2016-01-07 ┆ 4365.0   ┆ 4316.0     ┆ 4450.25 ┆ … ┆ 4351.5  ┆ 4316.75  ┆ 4293.25    ┆ 4365.0  │
│ 2016-01-08 ┆ 4339.75  ┆ 4337.0     ┆ 4298.0  ┆ … ┆ 4281.0  ┆ 4335.5   ┆ 4264.5     ┆ 4340.0  │
└────────────┴─

In [4]:
features = filter_valid_sessions(features)
print(features.select(["Session"]).tail(5))

shape: (5, 1)
┌────────────┐
│ Session    │
│ ---        │
│ date       │
╞════════════╡
│ 2026-02-20 │
│ 2026-02-23 │
│ 2026-02-24 │
│ 2026-02-25 │
│ 2026-02-26 │
└────────────┘


***

### **Defining Session Reference Open**
***
Reference open represents daily candle opening price and are defined as such:

$O_{Ref}(t)$ (`O_Ref_t`) = `O_Asia_t` -> For day `t`

Every M level bands will be placed relative to this level.

In [5]:
features = features.with_columns(
    pl.col("O_Asia").alias("O_Ref")
)
print(features.select(["Session", "O_Ref", "O_Asia"]).head(3))

shape: (3, 3)
┌────────────┬────────┬────────┐
│ Session    ┆ O_Ref  ┆ O_Asia │
│ ---        ┆ ---    ┆ ---    │
│ date       ┆ f64    ┆ f64    │
╞════════════╪════════╪════════╡
│ 2016-01-04 ┆ 4592.5 ┆ 4592.5 │
│ 2016-01-05 ┆ 4505.5 ┆ 4505.5 │
│ 2016-01-06 ┆ 4482.0 ┆ 4482.0 │
└────────────┴────────┴────────┘


***

### **Yang-Zhang Volatility**
***
We will be using Yang-Zhang volatility to help us predict the statistical levels, this levels will be used to capture intraday price movement.

You can use any volatility estimator but as previously stated in `02_feature_selection.ipynb` Yang-Zhang volatility fits our case the best

There will be 2 features we will be calculating with Yang-Zhang model:
- `sigma_historical`
- `sigma_today` (Pre NY movement as we don't want to leak NY data into the volatility model)

Yang-Zhang estimator are defined over $n$ trading days where each day $i$ has an overnight gap from the previous daily close $C_{i-1}$ to the open $O_i$:
$$\hat{\sigma}^2_{YZ} = \hat{\sigma}^2_o + k\,\hat{\sigma}^2_c + (1-k)\,\hat{\sigma}^2_{RS}$$

Where:

**Overnight Variance**:

$$\hat{\sigma}^2_o = \frac{1}{n-1} \sum_{i=1}^{n} \left( o_i - \bar{o} \right)^2
\quad \text{where} \quad
o_i = \ln\frac{O_i}{C_{i-1}}, \quad \bar{o} = \frac{1}{n}\sum_{i=1}^{n} o_i$$

**Open to Close Variance**:

$$\hat{\sigma}^2_c = \frac{1}{n-1} \sum_{i=1}^{n} \left( c_i - \bar{c} \right)^2
\quad \text{where} \quad
c_i = \ln\frac{C_i}{O_i}, \quad \bar{c} = \frac{1}{n}\sum_{i=1}^{n} c_i$$

**Rogers Satchell Variance**:

$$\hat{\sigma}^2_{RS} = \frac{1}{n} \sum_{i=1}^{n} \left[
    \ln\frac{H_i}{C_i}\ln\frac{H_i}{O_i} +
    \ln\frac{L_i}{C_i}\ln\frac{L_i}{O_i}
\right]$$

**Optimal Weight**:

$$k = \frac{0.34}{1.34 + \dfrac{n+1}{n-1}}$$

Note: $k$ is derived analytically and is not a tunable parameter, it depends only on $n$

*Limitations: `sigma_today` will underestimates because it only stops at london close since we dont want to leak NY session data, this is a limitation of our approach*
***
### **Sigma Historical (`sigma_historical`)**
Full OHLC including NY. Past days $d < t$ are fully resolved, using $C_{NY}(d)$ for a historical day is not leakage.

$$H^{full}(d) = \max(H_{Asia},\, H_{London},\, H_{NY}), \qquad L^{full}(d) = \min(L_{Asia},\, L_{London},\, L_{NY})$$

**Overnight Variance**:

$$\hat{\sigma}^2_o = \text{RollingVar}_N\!\left(\ln\frac{O_{Asia}(d)}{C_{NY}(d-1)}\right)$$

**Open to Close Variance**:

$$\hat{\sigma}^2_c = \text{RollingVar}_N\!\left(\ln\frac{C_{NY}(d)}{O_{Asia}(d)}\right)$$

**Rogers Satchell Variance**:

$$\hat{\sigma}^2_{RS} = \text{RollingMean}_N\!\left[\ln\frac{H^{full}}{C_{NY}}\ln\frac{H^{full}}{O_{Asia}} + \ln\frac{L^{full}}{C_{NY}}\ln\frac{L^{full}}{O_{Asia}}\right]$$

### **Sigma Today (`sigma_today`)**

Asia + London only, $C_{NY}(t)$ is forbidden.
The OC and RS components are approximated using the last observable price: $C_{London}(t)$.

$$H^{pre}(t) = \max(H_{Asia},\, H_{London}), \qquad L^{pre}(t) = \min(L_{Asia},\, L_{London})$$

**Overnight Variance**:

$$\hat{\sigma}^2_o = \text{RollingVar}_N\!\left(\ln\frac{O_{Asia}(d)}{C_{NY}(d-1)}\right)$$

**Open-to-Close Variance**:

$$\hat{\sigma}^2_{c,proxy} = \text{RollingVar}_N\!\left(\ln\frac{C_{London}(d)}{O_{Asia}(d)}\right)$$

**Rogers-Satchell Variance**:

$$\hat{\sigma}^2_{RS,pre} = \text{RollingMean}_N\!\left[\ln\frac{H^{pre}}{C_{London}}\ln\frac{H^{pre}}{O_{Asia}} + \ln\frac{L^{pre}}{C_{London}}\ln\frac{L^{pre}}{O_{Asia}}\right]$$


In [6]:
# Implementation for Yang Zhang estimator can be seen in `src/ohlc_dss_model/features/volatility.py`
features = yang_zhang(features, HISTORICAL_SPEC)
features = yang_zhang(features, TODAY_SPEC)

print(features.select(["Session", "Sigma_Historical", "Sigma_Today"]).drop_nulls().head(5))

# should be n nulls as it needs n days burn in period
print(f"Null count historical: {features['Sigma_Historical'].null_count()}")
print(f"Null count today: {features['Sigma_Today'].null_count()}")

shape: (5, 3)
┌────────────┬──────────────────┬─────────────┐
│ Session    ┆ Sigma_Historical ┆ Sigma_Today │
│ ---        ┆ ---              ┆ ---         │
│ date       ┆ f64              ┆ f64         │
╞════════════╪══════════════════╪═════════════╡
│ 2016-02-02 ┆ 0.019235         ┆ 0.011402    │
│ 2016-02-03 ┆ 0.019712         ┆ 0.011409    │
│ 2016-02-04 ┆ 0.01962          ┆ 0.011412    │
│ 2016-02-05 ┆ 0.019836         ┆ 0.010732    │
│ 2016-02-08 ┆ 0.020299         ┆ 0.011131    │
└────────────┴──────────────────┴─────────────┘
Null count historical: 20
Null count today: 20


### **Volatility Adjusted Excursion Bands**
***

Intraday price behavior can be simplified into 2 components:
- Adverse Excursion (AE) -> Counter directional movement from reference open $O_{Ref}$
- Favorable Excursion (FE) -> Co directional move following adverse excursion

This framework estimates AE and FE magnitudes based on recent volatility and regime to construct symmetric price bands relative to reference open $O_{Ref}$. This bands are then used to capture Pre NY session interaction for downstream modelling.

***

### **Volatility Scaling**
Excursion prices are normalized using Yang Zhang volatility:
$$\varepsilon^*_{AE}(d) = \frac{\varepsilon_{AE}(d)}{\hat{\sigma}_{YZ}(d)}, \qquad \varepsilon^*_{FE}(d) = \frac{\varepsilon_{FE}(d)}{\hat{\sigma}_{YZ}(d)}$$

This produce stationary quantities such that it can be used across time.

***

### **Direction Definition**

Normalized daily candle body are defined as such:
$$z_{body}(d) = \frac{|{ln}(C_{NY}(d) / O_{Ref}(d))|}{\hat{\sigma}_{YZ}(d)}$$
let $\tau(d)$ be a volatility adaptive threshold:
$$Z_{\sigma}(d) = \frac{\hat{\sigma}_{YZ}(d)}{{RollingMean}_{N}(\hat{\sigma}_{YZ}(d))}$$
$$\tau(d) = clip(\tau_0 * z_{\sigma}(d)^{-0.5}, \tau_{min}, \tau_{max})$$
Hence direction of a day is classified as such:
$${direction}(d) = \begin{cases}
{bullish} \qquad {z_{body}(d) > \tau(d) \land {C_{NY}(d)} > {O_{Asia}(d)}}\\
{bearish} \qquad {z_{body}(d) > \tau(d) \land {C_{NY}(d)} < {O_{Asia}(d)}}\\
{neutral} \qquad {z_{body}(d) < \tau(d)}
\end{cases}$$

Neutral here means day represents low volatility days
***

**Calculating $z_{body}(d)$**

In [7]:
features = features.with_columns(
    ((pl.col("C_New York") / pl.col("O_Ref")).log().abs() / pl.col("Sigma_Historical")).alias("_z_body")
)
print(features.select(["Session", "_z_body"]).drop_nulls().tail(3))

shape: (3, 2)
┌────────────┬──────────┐
│ Session    ┆ _z_body  │
│ ---        ┆ ---      │
│ date       ┆ f64      │
╞════════════╪══════════╡
│ 2026-02-24 ┆ 0.672282 │
│ 2026-02-25 ┆ 1.284166 │
│ 2026-02-26 ┆ 1.027076 │
└────────────┴──────────┘


In [8]:
features_clean = features.filter(
    pl.col("_z_body").is_finite() &
    pl.col("_z_body").is_not_null() &
    pl.col("Sigma_Historical").is_not_null()
)
features_clean.select(pl.col("_z_body")).describe()

statistic,_z_body
str,f64
"""count""",2531.0
"""null_count""",0.0
"""mean""",0.775136
"""std""",0.675915
"""min""",0.0
"""25%""",0.260548
"""50%""",0.623148
"""75%""",1.115166
"""max""",5.443358


With this values we can emphirically derive threshold parameters:
- $\tau_0$ -> Set to 75th percentile ($z_{body} = 1.11$), such that only upper quartile moves classifies as directional.
- $\tau_{min}$ -> Set to 25th percentile ($z_{body} = 0.26$), to prevent over sensitivity in high volatility regimes.
- $\tau_{max}$ -> Caps the upper tail to prevent the threshold from being too strict. ($z_{body} = 1.75$)

These parameters may be adjusted.
***

**Calculating $z_{\sigma}(d)$**

In [9]:
features = features.with_columns(
    (pl.col("Sigma_Historical") / pl.col("Sigma_Historical").rolling_mean(20)).alias("_z_sigma")
)

In [10]:
print(features.select(["_z_sigma"]).drop_nulls().describe())

shape: (9, 2)
┌────────────┬──────────┐
│ statistic  ┆ _z_sigma │
│ ---        ┆ ---      │
│ str        ┆ f64      │
╞════════════╪══════════╡
│ count      ┆ 2512.0   │
│ null_count ┆ 0.0      │
│ mean       ┆ 1.008405 │
│ std        ┆ 0.194507 │
│ min        ┆ 0.535591 │
│ 25%        ┆ 0.890311 │
│ 50%        ┆ 0.984999 │
│ 75%        ┆ 1.097798 │
│ max        ┆ 2.113973 │
└────────────┴──────────┘


***

**Calculating $\tau(d)$**

In [11]:
# As we discover earlier here are the threshold parameters
tau_0 = 1.11
tau_min = 0.26
tau_max = 1.75

In [12]:
features = features.with_columns(
    (tau_0 * (pl.col("_z_sigma") ** -0.5)).clip(tau_min, tau_max).alias("_tau")
)

In [17]:
print(features.select(["Session", "_tau", "_z_body"]).drop_nulls().tail(5))

shape: (5, 3)
┌────────────┬──────────┬──────────┐
│ Session    ┆ _tau     ┆ _z_body  │
│ ---        ┆ ---      ┆ ---      │
│ date       ┆ f64      ┆ f64      │
╞════════════╪══════════╪══════════╡
│ 2026-02-20 ┆ 1.044357 ┆ 0.694404 │
│ 2026-02-23 ┆ 1.053667 ┆ 0.716561 │
│ 2026-02-24 ┆ 1.071467 ┆ 0.672282 │
│ 2026-02-25 ┆ 1.076032 ┆ 1.284166 │
│ 2026-02-26 ┆ 1.078463 ┆ 1.027076 │
└────────────┴──────────┴──────────┘


***

**Deriving $direction(d)$**

In [ ]:
features = features.with_columns(
    pl.when((pl.col("_z_body") > pl.col("_tau")) & (pl.col("C_New York") > pl.col("O_Ref")))
    .then(pl.lit("bullish"))
    .when((pl.col("_z_body") > pl.col("_tau")) & (pl.col("C_New York") < pl.col("O_Ref")))
    .then(pl.lit("bearish"))
    .otherwise(pl.lit("neutral")).alias("Direction")
)

In [20]:
print(features.select(["Session", "Direction"]).drop_nulls().tail(5))

shape: (5, 2)
┌────────────┬───────────┐
│ Session    ┆ Direction │
│ ---        ┆ ---       │
│ date       ┆ str       │
╞════════════╪═══════════╡
│ 2026-02-20 ┆ neutral   │
│ 2026-02-23 ┆ neutral   │
│ 2026-02-24 ┆ neutral   │
│ 2026-02-25 ┆ bullish   │
│ 2026-02-26 ┆ neutral   │
└────────────┴───────────┘


In [25]:
print(features["Direction"].value_counts())

shape: (3, 2)
┌───────────┬───────┐
│ Direction ┆ count │
│ ---       ┆ ---   │
│ str       ┆ u32   │
╞═══════════╪═══════╡
│ bullish   ┆ 344   │
│ neutral   ┆ 1917  │
│ bearish   ┆ 290   │
└───────────┴───────┘


In practice, the parameter of $\tau(d) = 1.11$ is way too strict hence there is less directional classes, hence it would be wise to lower the parameter to somewhere between 50th percentile and 75th percentile. For our case we will be using $\tau(d) = 0.9$ 